# ReturnShield — Baseline Modeling

This notebook builds leakage-controlled baseline models for the ReturnShield transaction layer.

The notebook uses the modeling base table produced by the data preparation notebook. `InvoiceDate` is used for a real temporal holdout, so the test period represents future rows rather than a random sample from the same period.

## Modeling Design

The target variable is `is_return`, a line-item return/cancel proxy.

The modeling flow is organized around four principles:

- temporal split with future rows held out for testing
- customer and product history calculated from training data only
- smoothed history features for small groups
- top-k evaluation for risk ranking

The notebook also keeps key diagnostic checks inside the notebook. These checks are calculated live and are not exported as separate CSV files.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, f1_score
from sklearn.isotonic import IsotonicRegression

CURRENT_DIR = Path.cwd().resolve()
BASE_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

DATA_PROCESSED = BASE_DIR / "data" / "processed"
OUTPUT_DIR = BASE_DIR / "outputs"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_BASE_CANDIDATES = [
    DATA_PROCESSED / "online_retail_model_base.csv",
    BASE_DIR / "data" / "processed" / "online_retail_model_base_with_invoice_date.csv",
    CURRENT_DIR / "online_retail_model_base.csv",
    CURRENT_DIR / "data.csv",
]

MODEL_BASE_PATH = next((path for path in MODEL_BASE_CANDIDATES if path.exists()), None)
if MODEL_BASE_PATH is None:
    searched_paths = "\n".join(str(path) for path in MODEL_BASE_CANDIDATES)
    raise FileNotFoundError(f"Model base CSV file was not found. Run notebook 01 first. Searched paths:\n{searched_paths}")

METRICS_OUTPUT_PATH = OUTPUT_DIR / "online_retail_modeling_metrics.csv"
TOP_K_OUTPUT_PATH = OUTPUT_DIR / "online_retail_top_k_lift.csv"
TEST_SCORES_OUTPUT_PATH = OUTPUT_DIR / "online_retail_test_scores.csv"

pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", "{:.6f}".format)


In [2]:
# The notebook normally starts from online_retail_model_base.csv.
# The raw data fallback supports direct execution in this sandbox.
if MODEL_BASE_PATH.name == 'data.csv':
    raw_df = pd.read_csv(MODEL_BASE_PATH, encoding='ISO-8859-1')
    raw_df['InvoiceNo'] = raw_df['InvoiceNo'].astype(str)
    raw_df['InvoiceDate'] = pd.to_datetime(raw_df['InvoiceDate'], errors='coerce')
    raw_df['is_return'] = (raw_df['InvoiceNo'].str.startswith('C') | (raw_df['Quantity'] < 0)).astype(int)
    non_product_codes = ['POST', 'DOT', 'M', 'BANK CHARGES', 'AMAZONFEE', 'D', 'CRUK']
    model_df = raw_df[(~raw_df['StockCode'].isin(non_product_codes)) & (raw_df['UnitPrice'] > 0)].copy()
    model_df['log_unit_price'] = np.log1p(model_df['UnitPrice'])
    model_df['invoice_hour'] = model_df['InvoiceDate'].dt.hour
    model_df['invoice_dayofweek'] = model_df['InvoiceDate'].dt.dayofweek
    model_df['invoice_month'] = model_df['InvoiceDate'].dt.month
    model_df['evening_purchase'] = model_df['invoice_hour'].isin([17, 18, 19, 20]).astype(int)
    model_df['customer_history_available'] = model_df['CustomerID'].notna().astype(int)
    model_df = model_df[[
        'InvoiceDate', 'StockCode', 'Description', 'CustomerID', 'Country',
        'log_unit_price', 'invoice_hour', 'invoice_dayofweek', 'invoice_month',
        'evening_purchase', 'customer_history_available', 'is_return'
    ]].copy()
else:
    model_df = pd.read_csv(MODEL_BASE_PATH, parse_dates=['InvoiceDate'])

model_df = model_df.sort_values('InvoiceDate').reset_index(drop=True)

print('Source file:', MODEL_BASE_PATH)
print('Rows:', model_df.shape[0])
print('Columns:', model_df.shape[1])

model_df.head()

Source file: C:\Users\monster\Desktop\returnshield-ai\data\processed\online_retail_model_base.csv
Rows: 536704
Columns: 12


,InvoiceDate,StockCode,Description,CustomerID,Country,log_unit_price,invoice_hour,invoice_dayofweek,invoice_month,evening_purchase,customer_history_available,is_return
0,2010-12-01 08:26:00,85123A,WHITE HANGING HEART T-LIGHT HOLDER,17850.000000,United Kingdom,1.266948,8,2,12,0,1,0
1,2010-12-01 08:26:00,71053,WHITE METAL LANTERN,17850.000000,United Kingdom,1.479329,8,2,12,0,1,0
2,2010-12-01 08:26:00,84406B,CREAM CUPID HEARTS COAT HANGER,17850.000000,United Kingdom,1.321756,8,2,12,0,1,0
3,2010-12-01 08:26:00,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,17850.000000,United Kingdom,1.479329,8,2,12,0,1,0
4,2010-12-01 08:26:00,84029E,RED WOOLLY HOTTIE WHITE HEART.,17850.000000,United Kingdom,1.479329,8,2,12,0,1,0


## Temporal Split

The split follows the invoice date.

- train: before September 2011
- validation: September 2011
- calibration: October 2011
- test: November and December 2011

The test period is a future holdout. This makes the evaluation more conservative than a random split.

In [3]:
train_df = model_df[model_df['InvoiceDate'] < '2011-09-01'].copy()
val_df = model_df[(model_df['InvoiceDate'] >= '2011-09-01') & (model_df['InvoiceDate'] < '2011-10-01')].copy()
calib_df = model_df[(model_df['InvoiceDate'] >= '2011-10-01') & (model_df['InvoiceDate'] < '2011-11-01')].copy()
test_df = model_df[model_df['InvoiceDate'] >= '2011-11-01'].copy()

split_summary = pd.DataFrame({
    'split': ['train', 'validation', 'calibration', 'test'],
    'start_date': [train_df['InvoiceDate'].min(), val_df['InvoiceDate'].min(), calib_df['InvoiceDate'].min(), test_df['InvoiceDate'].min()],
    'end_date': [train_df['InvoiceDate'].max(), val_df['InvoiceDate'].max(), calib_df['InvoiceDate'].max(), test_df['InvoiceDate'].max()],
    'rows': [len(train_df), len(val_df), len(calib_df), len(test_df)],
    'return_cancel_rate': [
        train_df['is_return'].mean(),
        val_df['is_return'].mean(),
        calib_df['is_return'].mean(),
        test_df['is_return'].mean(),
    ],
})

split_summary

,split,start_date,end_date,rows,return_cancel_rate
0,train,2010-12-01 08:26:00,2011-08-31 17:31:00,317136,0.017374
1,validation,2011-09-01 08:25:00,2011-09-30 17:22:00,49836,0.015250
2,calibration,2011-10-02 10:32:00,2011-10-31 17:13:00,60214,0.018766
3,test,2011-11-01 08:16:00,2011-12-09 12:50:00,109518,0.012482


## Split Reading

The holdout period falls in November and December. This period may reflect seasonal transaction behavior.

The temporal result is therefore interpreted as a future-period stress test, not as a universal production estimate.

## Train-Only History Features

Customer and product return/cancel history is calculated from the training split.

Bayesian smoothing pulls low-count groups toward the global training rate. For training rows, leave-one-out logic removes the current row from its own group statistic. Validation, calibration, and test rows use training statistics only.

This creates a small train/serving difference: training rows use leave-one-out values, while future rows use full training history. In production, each new cart behaves like a future row because it is not part of the training history.

In [4]:
ALPHA = 50.0
CUSTOMER_RELIABLE_MIN = 10
PRODUCT_RELIABLE_MIN = 20

global_train_rate = train_df['is_return'].mean()


def _safe_group_stats(train_data, key):
    return (
        train_data.dropna(subset=[key])
        .groupby(key)['is_return']
        .agg(['sum', 'count'])
        .rename(columns={'sum': f'{key}_return_sum_train', 'count': f'{key}_count_train'})
    )


def add_history_features(train_data, other_data, alpha=ALPHA):
    train_data = train_data.copy()
    other_data = other_data.copy()

    customer_stats = _safe_group_stats(train_data, 'CustomerID')
    product_stats = _safe_group_stats(train_data, 'StockCode')

    train_data = train_data.merge(customer_stats, left_on='CustomerID', right_index=True, how='left')
    train_data = train_data.merge(product_stats, left_on='StockCode', right_index=True, how='left')

    other_data = other_data.merge(customer_stats, left_on='CustomerID', right_index=True, how='left')
    other_data = other_data.merge(product_stats, left_on='StockCode', right_index=True, how='left')

    for frame, is_train in [(train_data, True), (other_data, False)]:
        for prefix in ['CustomerID', 'StockCode']:
            sum_col = f'{prefix}_return_sum_train'
            count_col = f'{prefix}_count_train'
            frame[sum_col] = frame[sum_col].fillna(0)
            frame[count_col] = frame[count_col].fillna(0)

        if is_train:
            customer_sum = train_data['CustomerID_return_sum_train'] - train_data['is_return']
            customer_count = train_data['CustomerID_count_train'] - 1
            product_sum = train_data['StockCode_return_sum_train'] - train_data['is_return']
            product_count = train_data['StockCode_count_train'] - 1
        else:
            customer_sum = frame['CustomerID_return_sum_train']
            customer_count = frame['CustomerID_count_train']
            product_sum = frame['StockCode_return_sum_train']
            product_count = frame['StockCode_count_train']

        frame['customer_history_count_train'] = customer_count.clip(lower=0)
        frame['product_history_count_train'] = product_count.clip(lower=0)

        frame['customer_return_cancel_rate_unsmoothed'] = np.where(
            customer_count > 0,
            customer_sum / customer_count,
            global_train_rate,
        )
        frame['product_return_cancel_rate_unsmoothed'] = np.where(
            product_count > 0,
            product_sum / product_count,
            global_train_rate,
        )

        frame['customer_return_cancel_rate_train'] = (
            customer_sum + alpha * global_train_rate
        ) / (customer_count + alpha)
        frame['product_return_cancel_rate_train'] = (
            product_sum + alpha * global_train_rate
        ) / (product_count + alpha)

        frame['customer_history_reliable_train'] = (
            frame['customer_history_count_train'] >= CUSTOMER_RELIABLE_MIN
        ).astype(int)
        frame['product_history_reliable_train'] = (
            frame['product_history_count_train'] >= PRODUCT_RELIABLE_MIN
        ).astype(int)

    return train_data, other_data

train_features, val_features = add_history_features(train_df, val_df)
_, calib_features = add_history_features(train_df, calib_df)
_, test_features = add_history_features(train_df, test_df)

feature_check = pd.DataFrame({
    'split': ['train', 'validation', 'calibration', 'test'],
    'rows': [len(train_features), len(val_features), len(calib_features), len(test_features)],
    'mean_customer_rate': [
        train_features['customer_return_cancel_rate_train'].mean(),
        val_features['customer_return_cancel_rate_train'].mean(),
        calib_features['customer_return_cancel_rate_train'].mean(),
        test_features['customer_return_cancel_rate_train'].mean(),
    ],
    'mean_product_rate': [
        train_features['product_return_cancel_rate_train'].mean(),
        val_features['product_return_cancel_rate_train'].mean(),
        calib_features['product_return_cancel_rate_train'].mean(),
        test_features['product_return_cancel_rate_train'].mean(),
    ],
})

feature_check

,split,rows,mean_customer_rate,mean_product_rate
0,train,317136,0.019962,0.017395
1,validation,49836,0.019274,0.016989
2,calibration,60214,0.018762,0.017147
3,test,109518,0.019026,0.017012


## Feature Sets

Three feature sets are used.

`price_time` contains only price and transaction timing fields.

`transaction_context` adds country and customer-history availability.

`history_smoothed` adds train-only customer and product history features.

In [5]:
price_time_numeric = [
    'log_unit_price',
    'invoice_hour',
    'invoice_dayofweek',
    'invoice_month',
    'evening_purchase',
]

transaction_numeric = price_time_numeric + ['customer_history_available']
transaction_categorical = ['Country']

history_numeric_smoothed = transaction_numeric + [
    'customer_history_count_train',
    'product_history_count_train',
    'customer_history_reliable_train',
    'product_history_reliable_train',
    'customer_return_cancel_rate_train',
    'product_return_cancel_rate_train',
]

history_numeric_unsmoothed = transaction_numeric + [
    'customer_history_count_train',
    'product_history_count_train',
    'customer_history_reliable_train',
    'product_history_reliable_train',
    'customer_return_cancel_rate_unsmoothed',
    'product_return_cancel_rate_unsmoothed',
]

feature_sets = {
    'price_time': {'numeric': price_time_numeric, 'categorical': []},
    'transaction_context': {'numeric': transaction_numeric, 'categorical': transaction_categorical},
    'history_smoothed': {'numeric': history_numeric_smoothed, 'categorical': transaction_categorical},
    'history_unsmoothed': {'numeric': history_numeric_unsmoothed, 'categorical': transaction_categorical},
}

pd.DataFrame([
    {
        'feature_set': name,
        'numeric_features': len(spec['numeric']),
        'categorical_features': len(spec['categorical']),
    }
    for name, spec in feature_sets.items()
])

,feature_set,numeric_features,categorical_features
0,price_time,5,0
1,transaction_context,6,1
2,history_smoothed,12,1
3,history_unsmoothed,12,1


## Model Fitting Sample

The history features are calculated from the full training split.

Model fitting uses a stratified training sample to keep the notebook reproducible on standard machines. Validation, calibration, test, and all reported metrics are calculated on the full corresponding splits.

In [6]:
MAX_TRAIN_ROWS_FOR_MODEL = 50_000

if len(train_features) > MAX_TRAIN_ROWS_FOR_MODEL:
    train_model_df, _ = train_test_split(
        train_features,
        train_size=MAX_TRAIN_ROWS_FOR_MODEL,
        stratify=train_features['is_return'],
        random_state=42,
    )
else:
    train_model_df = train_features.copy()

model_fit_sample_summary = pd.DataFrame({
    'table': ['full_train_features', 'model_fit_sample'],
    'rows': [len(train_features), len(train_model_df)],
    'return_cancel_rate': [train_features['is_return'].mean(), train_model_df['is_return'].mean()],
})

model_fit_sample_summary

,table,rows,return_cancel_rate
0,full_train_features,317136,0.017374
1,model_fit_sample,50000,0.017380


## Modeling Utilities

The models use imputation inside pipelines. Raw `CustomerID`, `StockCode`, and `Description` are not model inputs.

In [7]:
def make_preprocess(numeric_features, categorical_features, scale_numeric=False):
    numeric_steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale_numeric:
        numeric_steps.append(('scaler', StandardScaler(with_mean=False)))

    transformers = [
        ('numeric', Pipeline(numeric_steps), numeric_features)
    ]

    if categorical_features:
        transformers.append((
            'categorical',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True))
            ]),
            categorical_features,
        ))

    return ColumnTransformer(transformers, remainder='drop', sparse_threshold=1.0)


def make_rf_model(numeric_features, categorical_features, max_depth=10, min_samples_leaf=50, n_estimators=8, seed=42):
    return Pipeline([
        ('preprocess', make_preprocess(numeric_features, categorical_features, scale_numeric=False)),
        ('model', RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            class_weight='balanced_subsample',
            random_state=seed,
            n_jobs=-1,
            max_samples=0.5,
        )),
    ])


def make_logistic_model(numeric_features, categorical_features, seed=42):
    return Pipeline([
        ('preprocess', make_preprocess(numeric_features, categorical_features, scale_numeric=True)),
        ('model', LogisticRegression(
            class_weight='balanced',
            max_iter=300,
            random_state=seed,
            solver='lbfgs',
        )),
    ])


def split_xy(data, numeric_features, categorical_features):
    cols = numeric_features + categorical_features
    return data[cols], data['is_return']


def predict_score(model, data, numeric_features, categorical_features):
    cols = numeric_features + categorical_features
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(data[cols])[:, 1]
    return model.decision_function(data[cols])


def metric_row(model_name, split_name, y_true, scores, threshold=None):
    if threshold is None:
        threshold = np.quantile(scores, 0.95)
    preds = (scores >= threshold).astype(int)
    return {
        'model': model_name,
        'split': split_name,
        'pr_auc': average_precision_score(y_true, scores),
        'roc_auc': roc_auc_score(y_true, scores) if y_true.nunique() == 2 else np.nan,
        'top_5pct_score_cutoff_from_validation': threshold,
        'precision_at_cutoff': precision_score(y_true, preds, zero_division=0),
        'recall_at_cutoff': recall_score(y_true, preds, zero_division=0),
        'f1_at_cutoff': f1_score(y_true, preds, zero_division=0),
        'base_rate': y_true.mean(),
    }


def evaluate_model(model_name, model, feature_spec, val_data, calib_data, test_data):
    numeric = feature_spec['numeric']
    categorical = feature_spec['categorical']
    val_scores = predict_score(model, val_data, numeric, categorical)
    calib_scores = predict_score(model, calib_data, numeric, categorical)
    test_scores = predict_score(model, test_data, numeric, categorical)
    threshold = np.quantile(val_scores, 0.95)
    rows = [
        metric_row(model_name, 'validation', val_data['is_return'], val_scores, threshold),
        metric_row(model_name, 'calibration', calib_data['is_return'], calib_scores, threshold),
        metric_row(model_name, 'test', test_data['is_return'], test_scores, threshold),
    ]
    return rows, val_scores, calib_scores, test_scores, threshold

## Hyperparameter Search

A compact grid is evaluated on the validation split for the Random Forest history model.

The grid is intentionally small because the notebook is a baseline rather than a full model-selection study.

In [8]:
grid_rows = []
for max_depth in [6, 10, 14]:
    for min_leaf in [50]:
        spec = feature_sets['history_smoothed']
        model = make_rf_model(
            spec['numeric'],
            spec['categorical'],
            max_depth=max_depth,
            min_samples_leaf=min_leaf,
            n_estimators=5,
            seed=42,
        )
        X_train, y_train = split_xy(train_model_df, spec['numeric'], spec['categorical'])
        model.fit(X_train, y_train)
        val_scores = predict_score(model, val_features, spec['numeric'], spec['categorical'])
        grid_rows.append({
            'max_depth': max_depth,
            'min_samples_leaf': min_leaf,
            'validation_pr_auc': average_precision_score(val_features['is_return'], val_scores),
        })

rf_grid = pd.DataFrame(grid_rows).sort_values('validation_pr_auc', ascending=False).reset_index(drop=True)
best_rf_params = rf_grid.iloc[0].to_dict()

rf_grid

,max_depth,min_samples_leaf,validation_pr_auc
0,14,50,0.062566
1,10,50,0.052974
2,6,50,0.051353


## Baseline Models

The notebook trains four baseline models:

- Random Forest with price and time features
- Random Forest with transaction context
- Random Forest with smoothed history features
- Logistic Regression with smoothed history features

An unsmoothed history model is also trained for a smoothing ablation.

In [9]:
models = {}
metric_rows = []
scores = {}
thresholds = {}

# Random Forest baselines
for model_name, feature_set_name in [
    ('rf_price_time', 'price_time'),
    ('rf_transaction_context', 'transaction_context'),
    ('rf_history_smoothed', 'history_smoothed'),
    ('rf_history_unsmoothed', 'history_unsmoothed'),
]:
    spec = feature_sets[feature_set_name]
    model = make_rf_model(
        spec['numeric'],
        spec['categorical'],
        max_depth=int(best_rf_params['max_depth']),
        min_samples_leaf=int(best_rf_params['min_samples_leaf']),
        n_estimators=8,
        seed=42,
    )
    X_train, y_train = split_xy(train_model_df, spec['numeric'], spec['categorical'])
    model.fit(X_train, y_train)
    rows, val_scores, calib_scores, test_scores, threshold = evaluate_model(
        model_name, model, spec, val_features, calib_features, test_features
    )
    models[model_name] = model
    scores[model_name] = {'validation': val_scores, 'calibration': calib_scores, 'test': test_scores}
    thresholds[model_name] = threshold
    metric_rows.extend(rows)

# Logistic baseline
spec = feature_sets['history_smoothed']
logistic_history = make_logistic_model(spec['numeric'], spec['categorical'], seed=42)
X_train, y_train = split_xy(train_model_df, spec['numeric'], spec['categorical'])
logistic_history.fit(X_train, y_train)
rows, val_scores, calib_scores, test_scores, threshold = evaluate_model(
    'logistic_history', logistic_history, spec, val_features, calib_features, test_features
)
models['logistic_history'] = logistic_history
scores['logistic_history'] = {'validation': val_scores, 'calibration': calib_scores, 'test': test_scores}
thresholds['logistic_history'] = threshold
metric_rows.extend(rows)

modeling_metrics = pd.DataFrame(metric_rows)
modeling_metrics.sort_values(['split', 'pr_auc'], ascending=[True, False])

,model,split,pr_auc,roc_auc,top_5pct_score_cutoff_from_validation,precision_at_cutoff,recall_at_cutoff,f1_at_cutoff,base_rate
13,logistic_history,calibration,0.082612,0.703636,0.785665,0.091830,0.264602,0.136343,0.018766
10,rf_history_unsmoothed,calibration,0.050362,0.681044,0.663479,0.076034,0.180531,0.107002,0.018766
7,rf_history_smoothed,calibration,0.049279,0.690846,0.654584,0.066131,0.160177,0.093613,0.018766
4,rf_transaction_context,calibration,0.039208,0.686276,0.565331,0.054140,0.195575,0.084804,0.018766
1,rf_price_time,calibration,0.024811,0.575013,0.561433,0.029820,0.115929,0.047438,0.018766
11,rf_history_unsmoothed,test,0.051929,0.751041,0.663479,0.065442,0.226774,0.101573,0.012482
8,rf_history_smoothed,test,0.049000,0.752751,0.654584,0.070681,0.222385,0.107269,0.012482
14,logistic_history,test,0.047891,0.748424,0.785665,0.063857,0.258230,0.102393,0.012482
5,rf_transaction_context,test,0.026243,0.697690,0.565331,0.030133,0.177030,0.051500,0.012482
2,rf_price_time,test,0.015155,0.561834,0.561433,0.015689,0.064375,0.025229,0.012482


## Model Selection Reading

The validation split is used for model selection.

The calibration and test splits are kept separate. The model-selection table also shows whether the leading validation model remains clearly ahead on the test split.

In [10]:
selection_candidates = ['rf_history_smoothed', 'logistic_history']
validation_selection = (
    modeling_metrics[
        (modeling_metrics['split'] == 'validation') &
        (modeling_metrics['model'].isin(selection_candidates))
    ]
    .sort_values('pr_auc', ascending=False)
    .reset_index(drop=True)
)

selected_model_name = validation_selection.loc[0, 'model']
runner_up_model_name = validation_selection.loc[1, 'model']

selected_test_pr = modeling_metrics.query(
    'model == @selected_model_name and split == "test"'
)['pr_auc'].iloc[0]
runner_up_test_pr = modeling_metrics.query(
    'model == @runner_up_model_name and split == "test"'
)['pr_auc'].iloc[0]

model_selection_reading = pd.DataFrame({
    'item': [
        'selected_model_by_validation_pr_auc',
        'runner_up_model_by_validation_pr_auc',
        'validation_pr_auc_gap',
        'test_pr_auc_gap',
    ],
    'value': [
        selected_model_name,
        runner_up_model_name,
        validation_selection.loc[0, 'pr_auc'] - validation_selection.loc[1, 'pr_auc'],
        selected_test_pr - runner_up_test_pr,
    ],
})

selection_model_note = pd.DataFrame({
    'reading': [
        'Small test gaps indicate a fragile model-selection result.'
    ]
})

display(validation_selection)
display(model_selection_reading)
display(selection_model_note)

,model,split,pr_auc,roc_auc,top_5pct_score_cutoff_from_validation,precision_at_cutoff,recall_at_cutoff,f1_at_cutoff,base_rate
0,rf_history_smoothed,validation,0.058558,0.746944,0.654584,0.070197,0.230263,0.107593,0.015250
1,logistic_history,validation,0.053275,0.755542,0.785665,0.081059,0.265789,0.124231,0.015250


,item,value
0,selected_model_by_validation_pr_auc,rf_history_smoothed
1,runner_up_model_by_validation_pr_auc,logistic_history
2,validation_pr_auc_gap,0.005283
3,test_pr_auc_gap,0.001109


,reading
0,Small test gaps indicate a fragile model-selec...


## Smoothing Ablation

The smoothed and unsmoothed history models are compared on the same temporal splits.

This check shows whether Bayesian smoothing changes the history-based baseline rather than only adding a named feature transformation.

In [11]:
smoothing_ablation = (
    modeling_metrics[
        modeling_metrics['model'].isin(['rf_history_smoothed', 'rf_history_unsmoothed'])
    ]
    .pivot_table(index='split', columns='model', values='pr_auc')
    .reset_index()
)
smoothing_ablation['smoothed_minus_unsmoothed'] = (
    smoothing_ablation['rf_history_smoothed'] - smoothing_ablation['rf_history_unsmoothed']
)

smoothing_ablation

model,split,rf_history_smoothed,rf_history_unsmoothed,smoothed_minus_unsmoothed
0,calibration,0.049279,0.050362,-0.001083
1,test,0.049000,0.051929,-0.002929
2,validation,0.058558,0.055059,0.003499


## Random Split Contrast

The random split diagnostic is calculated inside the notebook and is not used for final model selection.

It shows how much easier the task becomes when rows from the same periods and customer/product groups are mixed across splits.

In [12]:
random_train, random_temp = train_test_split(
    model_df,
    test_size=0.30,
    stratify=model_df['is_return'],
    random_state=42,
)
random_val, random_test = train_test_split(
    random_temp,
    test_size=0.50,
    stratify=random_temp['is_return'],
    random_state=42,
)

random_train_features, random_val_features = add_history_features(random_train, random_val)
_, random_test_features = add_history_features(random_train, random_test)

spec = feature_sets['history_smoothed']
random_rf = make_rf_model(
    spec['numeric'],
    spec['categorical'],
    max_depth=int(best_rf_params['max_depth']),
    min_samples_leaf=int(best_rf_params['min_samples_leaf']),
    n_estimators=8,
    seed=42,
)
if len(random_train_features) > MAX_TRAIN_ROWS_FOR_MODEL:
    random_train_model_df, _ = train_test_split(
        random_train_features,
        train_size=MAX_TRAIN_ROWS_FOR_MODEL,
        stratify=random_train_features['is_return'],
        random_state=42,
    )
else:
    random_train_model_df = random_train_features.copy()

X_random_train, y_random_train = split_xy(random_train_model_df, spec['numeric'], spec['categorical'])
random_rf.fit(X_random_train, y_random_train)
random_test_scores = predict_score(random_rf, random_test_features, spec['numeric'], spec['categorical'])

temporal_test_pr = modeling_metrics.query(
    'model == "rf_history_smoothed" and split == "test"'
)['pr_auc'].iloc[0]
random_test_pr = average_precision_score(random_test_features['is_return'], random_test_scores)

random_vs_temporal = pd.DataFrame({
    'evaluation': ['random_split', 'temporal_holdout'],
    'test_pr_auc': [random_test_pr, temporal_test_pr],
    'reading': [
        'same-period row mixing',
        'future-period holdout',
    ],
})

random_vs_temporal

,evaluation,test_pr_auc,reading
0,random_split,0.120939,same-period row mixing
1,temporal_holdout,0.049000,future-period holdout


## Feature Importance Diagnostic

The Random Forest history model is inspected to understand which signals drive the ranking.

The `customer_history_available` feature is a data-availability signal. High importance for this feature reflects a registered-versus-unregistered customer artifact in the dataset, not a pure behavioral pattern.

In [13]:
rf_history_model = models['rf_history_smoothed']
preprocess = rf_history_model.named_steps['preprocess']
feature_names = preprocess.get_feature_names_out()
importances = rf_history_model.named_steps['model'].feature_importances_

feature_importance = (
    pd.DataFrame({'feature': feature_names, 'importance': importances})
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
)

availability_rows = feature_importance[
    feature_importance['feature'].str.contains('customer_history_available', regex=False)
]

feature_importance_reading = pd.DataFrame({
    'metric': ['customer_history_available_importance'],
    'value': [availability_rows['importance'].sum() if len(availability_rows) else 0.0],
})

display(feature_importance.head(15))
display(feature_importance_reading)

,feature,importance
0,numeric__customer_return_cancel_rate_train,0.376427
1,numeric__product_return_cancel_rate_train,0.273602
2,numeric__customer_history_count_train,0.105641
3,numeric__customer_history_available,0.063094
4,numeric__log_unit_price,0.040269
5,numeric__product_history_count_train,0.038728
6,numeric__invoice_hour,0.032771
7,numeric__invoice_dayofweek,0.030846
8,numeric__invoice_month,0.020519
9,numeric__evening_purchase,0.006027


,metric,value
0,customer_history_available_importance,0.063094


## Seed Stability

The Random Forest history model is repeated with multiple random seeds.

This check estimates how much the temporal PR-AUC changes because of model randomness.

In [14]:
seed_rows = []
for seed in [11, 22, 33]:
    spec = feature_sets['history_smoothed']
    seed_model = make_rf_model(
        spec['numeric'],
        spec['categorical'],
        max_depth=int(best_rf_params['max_depth']),
        min_samples_leaf=int(best_rf_params['min_samples_leaf']),
        n_estimators=5,
        seed=seed,
    )
    X_train, y_train = split_xy(train_model_df, spec['numeric'], spec['categorical'])
    seed_model.fit(X_train, y_train)
    seed_scores = predict_score(seed_model, test_features, spec['numeric'], spec['categorical'])
    seed_rows.append({
        'seed': seed,
        'test_pr_auc': average_precision_score(test_features['is_return'], seed_scores),
    })

seed_stability = pd.DataFrame(seed_rows)
seed_stability_summary = seed_stability['test_pr_auc'].agg(['mean', 'std', 'min', 'max']).to_frame().T

selection_fragility = pd.DataFrame({
    'metric': ['rf_vs_logistic_test_pr_auc_gap', 'rf_seed_std'],
    'value': [
        modeling_metrics.query('model == "rf_history_smoothed" and split == "test"')['pr_auc'].iloc[0]
        - modeling_metrics.query('model == "logistic_history" and split == "test"')['pr_auc'].iloc[0],
        seed_stability_summary['std'].iloc[0],
    ],
})

display(seed_stability)
display(seed_stability_summary)
display(selection_fragility)

,seed,test_pr_auc
0,11,0.038482
1,22,0.053157
2,33,0.046952


,mean,std,min,max
test_pr_auc,0.046197,0.007367,0.038482,0.053157


,metric,value
0,rf_vs_logistic_test_pr_auc_gap,0.001109
1,rf_seed_std,0.007367


## Calibration Check

The selected model score is calibrated with isotonic regression using the calibration split.

The calibration table uses equal-frequency deciles on the temporal test set. Non-monotonic observed rates indicate that calibrated scores are more reliable as relative ranking signals than as exact probabilities.

In [15]:
selected_model = models[selected_model_name]
selected_val_scores = scores[selected_model_name]['validation']
selected_calib_scores = scores[selected_model_name]['calibration']
selected_test_scores = scores[selected_model_name]['test']

iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(selected_calib_scores, calib_features['is_return'])
selected_test_calibrated = iso.transform(selected_test_scores)

calibration_df = pd.DataFrame({
    'score': selected_test_scores,
    'calibrated_score': selected_test_calibrated,
    'is_return': test_features['is_return'].values,
})
calibration_df['score_decile'] = pd.qcut(
    calibration_df['score'].rank(method='first'),
    10,
    labels=False,
) + 1

calibration_bins = (
    calibration_df.groupby('score_decile')
    .agg(
        row_count=('is_return', 'count'),
        min_score=('score', 'min'),
        max_score=('score', 'max'),
        mean_score=('score', 'mean'),
        mean_calibrated_score=('calibrated_score', 'mean'),
        observed_return_cancel_rate=('is_return', 'mean'),
    )
    .reset_index()
)

calibration_monotonicity = pd.DataFrame({
    'metric': ['downward_steps_in_observed_rate'],
    'value': [(calibration_bins['observed_return_cancel_rate'].diff().dropna() < 0).sum()],
})

calibration_reading = pd.DataFrame({
    'reading': [
        'Temporal calibration is not monotonic when downward steps are present. The score is treated as a ranking signal in the MVP.'
    ]
})

display(calibration_bins)
display(calibration_monotonicity)
display(calibration_reading)

,score_decile,row_count,min_score,max_score,mean_score,mean_calibrated_score,observed_return_cancel_rate
0,1,10952,0.000000,0.092098,0.071838,0.001330,0.001187
1,2,10952,0.092098,0.119136,0.105729,0.002322,0.002283
2,3,10952,0.119136,0.154754,0.139595,0.007962,0.006026
3,4,10951,0.154754,0.186391,0.172252,0.012473,0.005388
4,5,10952,0.186391,0.224300,0.202941,0.012840,0.007852
5,6,10952,0.224300,0.263721,0.244954,0.014871,0.009222
6,7,10951,0.263721,0.313146,0.289079,0.019888,0.009497
7,8,10952,0.313165,0.381545,0.344359,0.022766,0.015248
8,9,10952,0.381565,0.511495,0.432296,0.029692,0.018901
9,10,10952,0.511495,0.928822,0.646382,0.050312,0.049215


,metric,value
0,downward_steps_in_observed_rate,1


,reading
0,Temporal calibration is not monotonic when dow...


## Top-k Risk Ranking

ReturnShield uses risk scores to rank items for review or intervention.

Top-k precision, recall, and lift are more aligned with this use case than a single hard classification threshold.

In [16]:
def top_k_table(y_true, scores, model_name, ks=(0.01, 0.05, 0.10)):
    data = pd.DataFrame({'y': y_true.values, 'score': scores})
    data = data.sort_values('score', ascending=False).reset_index(drop=True)
    base_rate = data['y'].mean()
    rows = []
    for k in ks:
        n = max(1, int(len(data) * k))
        top = data.head(n)
        precision = top['y'].mean()
        recall = top['y'].sum() / data['y'].sum()
        rows.append({
            'model': model_name,
            'top_k_percent': int(k * 100),
            'rows_in_top_k': n,
            'precision_at_k': precision,
            'recall_at_k': recall,
            'test_base_rate': base_rate,
            'lift_at_k': precision / base_rate if base_rate > 0 else np.nan,
        })
    return pd.DataFrame(rows)

top_k_lift = top_k_table(test_features['is_return'], selected_test_scores, selected_model_name)

top_k_lift

,model,top_k_percent,rows_in_top_k,precision_at_k,recall_at_k,test_base_rate,lift_at_k
0,rf_history_smoothed,1,1095,0.106849,0.085589,0.012482,8.560295
1,rf_history_smoothed,5,5475,0.066484,0.266277,0.012482,5.326406
2,rf_history_smoothed,10,10951,0.049219,0.394294,0.012482,3.943229


## Final Outputs

Only three project-level outputs are exported:

- model metrics
- top-k lift table
- test scores

Diagnostic checks remain inside the notebook and are calculated during execution.

In [17]:
test_scores = test_features[[
    'InvoiceDate',
    'StockCode',
    'CustomerID',
    'Country',
    'log_unit_price',
    'invoice_hour',
    'invoice_dayofweek',
    'invoice_month',
    'evening_purchase',
    'customer_history_available',
    'is_return',
]].copy()

test_scores['selected_model'] = selected_model_name
test_scores['selected_model_score'] = selected_test_scores
test_scores['selected_model_calibrated_score'] = selected_test_calibrated

test_scores['rf_history_smoothed_score'] = scores['rf_history_smoothed']['test']
test_scores['logistic_history_score'] = scores['logistic_history']['test']

modeling_metrics.to_csv(METRICS_OUTPUT_PATH, index=False)
top_k_lift.to_csv(TOP_K_OUTPUT_PATH, index=False)
test_scores.to_csv(TEST_SCORES_OUTPUT_PATH, index=False)

print('Saved:', METRICS_OUTPUT_PATH)
print('Saved:', TOP_K_OUTPUT_PATH)
print('Saved:', TEST_SCORES_OUTPUT_PATH)

Saved: C:\Users\monster\Desktop\returnshield-ai\outputs\online_retail_modeling_metrics.csv
Saved: C:\Users\monster\Desktop\returnshield-ai\outputs\online_retail_top_k_lift.csv
Saved: C:\Users\monster\Desktop\returnshield-ai\outputs\online_retail_test_scores.csv


## Notebook Summary

This notebook evaluates return/cancel risk with a temporal holdout.

The temporal result is more conservative than a random split result. The notebook keeps the random split comparison, seed stability, smoothing ablation, model-selection fragility, feature-importance artifact check, and calibration monotonicity check inside the notebook as live calculations.

The final score is used as a relative risk ranking signal. The temporal calibration check shows that calibrated values are not interpreted as exact return probabilities in the MVP.